In [1]:
import pandas as pd

hotel = pd.read_csv(
    "processed_data/hotel_bookings_cleaned.csv"
)

hotel.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,0.0,0,Transient,0.0,0,0,Check-Out,01/07/2015
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,0.0,0,Transient,0.0,0,0,Check-Out,01/07/2015
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,0.0,0,Transient,75.0,0,0,Check-Out,02/07/2015
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304.0,0,Transient,75.0,0,0,Check-Out,02/07/2015
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240.0,0,Transient,98.0,0,1,Check-Out,03/07/2015


In [2]:
hotel.shape

(87369, 31)

In [3]:
hotel.isnull().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
days_in_waiting_list              0
customer_type                     0
adr                               0
required_car_parking_spaces 

In [4]:
hotel.drop(
    columns=[
        "reservation_status",
        "reservation_status_date"
    ],
    inplace=True
)

In [5]:
hotel["is_canceled"].value_counts()

hotel["is_canceled"].value_counts(normalize=True)

is_canceled
0    0.725028
1    0.274972
Name: proportion, dtype: float64

In [6]:
hotel.nunique().sort_values()

hotel                                2
is_canceled                          2
is_repeated_guest                    2
arrival_date_year                    3
deposit_type                         3
customer_type                        4
distribution_channel                 5
required_car_parking_spaces          5
meal                                 5
babies                               5
children                             5
total_of_special_requests            6
market_segment                       8
reserved_room_type                  10
arrival_date_month                  12
assigned_room_type                  12
adults                              14
previous_cancellations              15
stays_in_weekend_nights             17
booking_changes                     21
arrival_date_day_of_month           31
stays_in_week_nights                35
arrival_date_week_number            53
previous_bookings_not_canceled      73
days_in_waiting_list               128
country                  

In [8]:
import os
import numpy as np
import pandas as pd


# ============================================================
# 1. CREATE A COPY
# ============================================================

hotel = hotel.copy()

print("Original dataset shape:", hotel.shape)


# ============================================================
# 2. TOTAL NUMBER OF NIGHTS
# ============================================================

hotel["total_nights"] = (
    hotel["stays_in_weekend_nights"]
    + hotel["stays_in_week_nights"]
)


# ============================================================
# 3. TOTAL NUMBER OF GUESTS
# ============================================================

hotel["total_guests"] = (
    hotel["adults"]
    + hotel["children"]
    + hotel["babies"]
)


# ============================================================
# 4. FAMILY BOOKING INDICATOR
# ============================================================

hotel["is_family"] = (
    (hotel["children"] > 0)
    | (hotel["babies"] > 0)
).astype(int)


# ============================================================
# 5. ROOM CHANGE INDICATOR
# ============================================================

hotel["room_changed"] = (
    hotel["reserved_room_type"]
    != hotel["assigned_room_type"]
).astype(int)


# ============================================================
# 6. PREVIOUS CANCELLATION RATIO
# ============================================================

previous_total_bookings = (
    hotel["previous_cancellations"]
    + hotel["previous_bookings_not_canceled"]
)

hotel["previous_cancellation_ratio"] = np.where(
    previous_total_bookings > 0,
    hotel["previous_cancellations"]
    / previous_total_bookings,
    0
)


# ============================================================
# 7. PREVIOUS BOOKING HISTORY INDICATOR
# ============================================================

hotel["has_previous_booking"] = (
    previous_total_bookings > 0
).astype(int)


# ============================================================
# 8. BOOKED THROUGH AGENT INDICATOR
# ============================================================

if "agent" in hotel.columns:

    hotel["booked_through_agent"] = (
        hotel["agent"].fillna(0) != 0
    ).astype(int)

elif "booked_through_agent" not in hotel.columns:

    hotel["booked_through_agent"] = 0


# ============================================================
# 9. COMPANY BOOKING INDICATOR
# ============================================================

if "company" in hotel.columns:

    hotel["company_booking"] = (
        hotel["company"].fillna(0) != 0
    ).astype(int)


# ============================================================
# 10. ESTIMATED BOOKING VALUE
# ============================================================

hotel["estimated_booking_value"] = (
    hotel["adr"]
    * hotel["total_nights"]
)


# ============================================================
# 11. AVERAGE DAILY RATE PER GUEST
# ============================================================

hotel["adr_per_guest"] = np.where(
    hotel["total_guests"] > 0,
    hotel["adr"] / hotel["total_guests"],
    0
)


# ============================================================
# 12. LEAD-TIME CATEGORY
# ============================================================

lead_time_bins = [
    -1,
    7,
    30,
    90,
    180,
    np.inf
]

lead_time_labels = [
    "Last Minute",
    "Short",
    "Medium",
    "Long",
    "Very Long"
]

hotel["lead_time_category"] = pd.cut(
    hotel["lead_time"],
    bins=lead_time_bins,
    labels=lead_time_labels
)


# ============================================================
# 13. WEEKEND-STAY INDICATOR
# ============================================================

hotel["has_weekend_stay"] = (
    hotel["stays_in_weekend_nights"] > 0
).astype(int)


# ============================================================
# 14. LONG-STAY INDICATOR
# ============================================================

hotel["is_long_stay"] = (
    hotel["total_nights"] >= 7
).astype(int)


# ============================================================
# 15. SPECIAL REQUEST INDICATOR
# ============================================================

hotel["has_special_requests"] = (
    hotel["total_of_special_requests"] > 0
).astype(int)


# ============================================================
# 16. BOOKING CHANGE INDICATOR
# ============================================================

hotel["booking_was_changed"] = (
    hotel["booking_changes"] > 0
).astype(int)


# ============================================================
# 17. WAITING-LIST INDICATOR
# ============================================================

hotel["was_on_waiting_list"] = (
    hotel["days_in_waiting_list"] > 0
).astype(int)


# ============================================================
# 18. PARKING REQUIRED INDICATOR
# ============================================================

hotel["parking_required"] = (
    hotel["required_car_parking_spaces"] > 0
).astype(int)


# ============================================================
# 19. ARRIVAL MONTH NUMBER
# ============================================================

month_mapping = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}


if "arrival_date_month" in hotel.columns:

    hotel["arrival_month_number"] = (
        hotel["arrival_date_month"]
        .astype(str)
        .str.strip()
        .map(month_mapping)
    )

elif "arrival_month_number" in hotel.columns:

    hotel["arrival_month_number"] = pd.to_numeric(
        hotel["arrival_month_number"],
        errors="coerce"
    )

else:

    raise KeyError(
        "The dataset must contain either "
        "'arrival_date_month' or 'arrival_month_number'."
    )


# ============================================================
# 20. CYCLICAL MONTH ENCODING
# ============================================================

hotel["arrival_month_sin"] = np.sin(
    2 * np.pi
    * hotel["arrival_month_number"]
    / 12
)

hotel["arrival_month_cos"] = np.cos(
    2 * np.pi
    * hotel["arrival_month_number"]
    / 12
)


# ============================================================
# 21. COMPLETE ARRIVAL DATE
# ============================================================

hotel["arrival_date"] = pd.to_datetime(
    {
        "year": hotel["arrival_date_year"],
        "month": hotel["arrival_month_number"],
        "day": hotel["arrival_date_day_of_month"]
    },
    errors="coerce"
)


# ============================================================
# 22. ARRIVAL SEASON
# ============================================================

def assign_season(month):

    if pd.isna(month):
        return "Unknown"

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Spring"

    elif month in [6, 7, 8]:
        return "Summer"

    else:
        return "Autumn"


hotel["arrival_season"] = (
    hotel["arrival_month_number"]
    .apply(assign_season)
)


# ============================================================
# 23. ADULT-ONLY BOOKING INDICATOR
# ============================================================

hotel["adult_only_booking"] = (
    (hotel["adults"] > 0)
    & (hotel["children"] == 0)
    & (hotel["babies"] == 0)
).astype(int)


# ============================================================
# 24. LONG LEAD TIME WITH NO DEPOSIT
# ============================================================

hotel["long_lead_no_deposit"] = (
    (hotel["lead_time"] > 90)
    & (hotel["deposit_type"] == "No Deposit")
).astype(int)


# ============================================================
# 25. REPLACE INFINITE VALUES
# ============================================================

numeric_columns = hotel.select_dtypes(
    include=np.number
).columns

hotel[numeric_columns] = hotel[numeric_columns].replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 26. HANDLE MISSING VALUES CREATED DURING ENGINEERING
# ============================================================

hotel["arrival_month_number"] = (
    hotel["arrival_month_number"]
    .fillna(hotel["arrival_month_number"].median())
)

hotel["arrival_date"] = pd.to_datetime(
    {
        "year": hotel["arrival_date_year"],
        "month": hotel["arrival_month_number"].astype(int),
        "day": hotel["arrival_date_day_of_month"]
    },
    errors="coerce"
)

hotel = hotel.dropna(
    subset=["arrival_date"]
)


numeric_columns = hotel.select_dtypes(
    include=np.number
).columns

for column in numeric_columns:

    if hotel[column].isnull().sum() > 0:

        hotel[column] = hotel[column].fillna(
            hotel[column].median()
        )


categorical_columns = hotel.select_dtypes(
    include=["object", "category"]
).columns

for column in categorical_columns:

    hotel[column] = (
        hotel[column]
        .astype("object")
        .fillna("Unknown")
    )


# ============================================================
# 27. REMOVE REDUNDANT COLUMNS
# ============================================================

hotel = hotel.drop(
    columns=[
        "arrival_date_month",
        "agent",
        "company"
    ],
    errors="ignore"
)


# ============================================================
# 28. REMOVE DUPLICATES
# ============================================================

hotel = hotel.drop_duplicates().reset_index(
    drop=True
)


# ============================================================
# 29. DISPLAY ENGINEERED FEATURES
# ============================================================

engineered_features = [
    "total_nights",
    "total_guests",
    "is_family",
    "room_changed",
    "previous_cancellation_ratio",
    "has_previous_booking",
    "booked_through_agent",
    "estimated_booking_value",
    "adr_per_guest",
    "lead_time_category",
    "has_weekend_stay",
    "is_long_stay",
    "has_special_requests",
    "booking_was_changed",
    "was_on_waiting_list",
    "parking_required",
    "arrival_month_number",
    "arrival_month_sin",
    "arrival_month_cos",
    "arrival_date",
    "arrival_season",
    "adult_only_booking",
    "long_lead_no_deposit"
]

engineered_features = [
    column
    for column in engineered_features
    if column in hotel.columns
]

print("\nEngineered features:")
print(engineered_features)

display(
    hotel[engineered_features].head()
)


# ============================================================
# 30. FINAL VALIDATION
# ============================================================

print("\nFinal dataset shape:")
print(hotel.shape)

print("\nTotal missing values:")
print(hotel.isnull().sum().sum())

print("\nTotal duplicate rows:")
print(hotel.duplicated().sum())

print("\nTotal infinite values:")
print(
    np.isinf(
        hotel.select_dtypes(include=np.number)
    ).sum().sum()
)

print("\nTarget distribution:")
print(
    hotel["is_canceled"]
    .value_counts()
)

print("\nTarget distribution percentage:")
print(
    hotel["is_canceled"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ============================================================
# 31. SAVE FEATURE-ENGINEERED DATASET
# ============================================================

os.makedirs(
    "processed_data",
    exist_ok=True
)

output_path = (
    "processed_data/"
    "hotel_bookings_feature_engineered.csv"
)

hotel.to_csv(
    output_path,
    index=False
)

print(
    "\nFeature-engineered dataset saved successfully."
)

print("File path:", output_path)
print("Saved dataset shape:", hotel.shape)


# ============================================================
# 32. RELOAD SAVED DATASET TO CONFIRM
# ============================================================

hotel_check = pd.read_csv(
    output_path
)

print("\nReloaded dataset shape:")
print(hotel_check.shape)

print("\nReloaded missing values:")
print(hotel_check.isnull().sum().sum())

display(hotel_check.head())


Original dataset shape: (87369, 29)

Engineered features:
['total_nights', 'total_guests', 'is_family', 'room_changed', 'previous_cancellation_ratio', 'has_previous_booking', 'booked_through_agent', 'estimated_booking_value', 'adr_per_guest', 'lead_time_category', 'has_weekend_stay', 'is_long_stay', 'has_special_requests', 'booking_was_changed', 'was_on_waiting_list', 'parking_required', 'arrival_month_number', 'arrival_month_sin', 'arrival_month_cos', 'arrival_date', 'arrival_season', 'adult_only_booking', 'long_lead_no_deposit']


,total_nights,total_guests,is_family,room_changed,previous_cancellation_ratio,has_previous_booking,booked_through_agent,estimated_booking_value,adr_per_guest,lead_time_category,...,booking_was_changed,was_on_waiting_list,parking_required,arrival_month_number,arrival_month_sin,arrival_month_cos,arrival_date,arrival_season,adult_only_booking,long_lead_no_deposit
0,0,2.0,0,0,0.0,0,0,0.0,0.0,Very Long,...,1,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,1
1,0,2.0,0,0,0.0,0,0,0.0,0.0,Very Long,...,1,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,1
2,1,1.0,0,1,0.0,0,0,75.0,75.0,Last Minute,...,0,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,0
3,1,1.0,0,0,0.0,0,1,75.0,75.0,Short,...,0,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,0
4,2,2.0,0,0,0.0,0,1,196.0,49.0,Short,...,0,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,0



Final dataset shape:
(87097, 50)

Total missing values:
0

Total duplicate rows:
0

Total infinite values:
0

Target distribution:
is_canceled
0    63336
1    23761
Name: count, dtype: int64

Target distribution percentage:
is_canceled
0    72.72
1    27.28
Name: proportion, dtype: float64

Feature-engineered dataset saved successfully.
File path: processed_data/hotel_bookings_feature_engineered.csv
Saved dataset shape: (87097, 50)

Reloaded dataset shape:
(87097, 50)

Reloaded missing values:
0


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,booking_was_changed,was_on_waiting_list,parking_required,arrival_month_number,arrival_month_sin,arrival_month_cos,arrival_date,arrival_season,adult_only_booking,long_lead_no_deposit
0,Resort Hotel,0,342,2015,27,1,0,0,2,0.0,...,1,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,1
1,Resort Hotel,0,737,2015,27,1,0,0,2,0.0,...,1,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,1
2,Resort Hotel,0,7,2015,27,1,0,1,1,0.0,...,0,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,0
3,Resort Hotel,0,13,2015,27,1,0,1,1,0.0,...,0,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,0
4,Resort Hotel,0,14,2015,27,1,0,2,2,0.0,...,0,0,0,7,-0.5,-0.866025,2015-07-01,Summer,1,0
